# Image EDA

In [ ]:
from pathlib import Path
from itertools import combinations
import io
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
import matplotlib.pyplot as plt
from IPython.display import display, Image as DisplayImage

BASE_DIR = Path.cwd()
if not (BASE_DIR / "data").is_dir():
    BASE_DIR = BASE_DIR / "analyse"
DATA_DIR = BASE_DIR / "data"
if not DATA_DIR.is_dir():
    raise FileNotFoundError("Run this notebook from the project root or analyse folder.")
print("Dataset:", DATA_DIR.resolve())

def show_figure(fig):
    """Display and retain the figure inside the notebook, without a browser."""
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", bbox_inches="tight", dpi=110)
    plt.close(fig)
    display(DisplayImage(data=buffer.getvalue()))

## 1. Load images and labels
Each subfolder is a class. Convert each image to RGB and load its original pixel matrix. Do not resize before checking exact duplicates.

In [ ]:
extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
classes = sorted(folder.name for folder in DATA_DIR.iterdir() if folder.is_dir())
image_matrices, paths, labels, unreadable, counts = [], [], [], [], []
for label_id, name in enumerate(classes):
    files = sorted(p for p in (DATA_DIR / name).rglob("*")
                   if p.is_file() and p.suffix.lower() in extensions)
    loaded = 0
    for path in files:
        try:
            with Image.open(path) as source:
                matrix = np.array(ImageOps.exif_transpose(source).convert("RGB"))
        except (OSError, ValueError, Image.DecompressionBombError) as error:
            unreadable.append({"path": path.relative_to(DATA_DIR).as_posix(), "error": str(error)})
            continue
        image_matrices.append(matrix)
        paths.append(path.relative_to(DATA_DIR).as_posix())
        labels.append(label_id)
        loaded += 1
    counts.append({"label_id": label_id, "class": name, "files": len(files),
                   "loaded": loaded, "unreadable": len(files) - loaded})
if not image_matrices:
    raise ValueError("No readable images found.")
class_counts = pd.DataFrame(counts)
display(class_counts)
print(f"Total: {len(image_matrices)} readable images, {len(classes)} classes, {len(unreadable)} unreadable.")

## 2. Transform into matrices and inspect a small example
`X[i]` is the RGB matrix for image `i`; `y[i]` is its class number. Each pixel contains three values (red, green, blue) between 0 and 255. If dimensions vary, retain the original matrices in a list so exact comparison remains accurate.

In [ ]:
y = np.asarray(labels, dtype=np.int64)
shapes = {matrix.shape for matrix in image_matrices}
X = np.stack(image_matrices) if len(shapes) == 1 else image_matrices
print("X shape:", X.shape if isinstance(X, np.ndarray) else "List of original matrices with varying shapes")
print("y shape:", y.shape)
print("First image:", paths[0])
print("First matrix shape:", image_matrices[0].shape, "dtype:", image_matrices[0].dtype)
print("Top-left 3 x 3 pixels (each entry is [R, G, B]):")
print(image_matrices[0][:3, :3, :])
fig, axes = plt.subplots(1, min(3, len(image_matrices)), figsize=(10, 3), squeeze=False)
for index, ax in enumerate(axes[0]):
    ax.imshow(image_matrices[index])
    ax.set_title(f"Image {index}: {classes[y[index]]}")
    ax.axis("off")
show_figure(fig)

In [ ]:
from pathlib import Path
import pandas as pd
from PIL import Image, ImageOps
from IPython.display import display

_dimensions_root = Path.cwd()
if not (_dimensions_root / "data").is_dir():
    _dimensions_root = _dimensions_root / "analyse"
_dimensions_data = _dimensions_root / "data"
if not _dimensions_data.is_dir():
    raise FileNotFoundError("Open the notebook from the project root or analyse folder.")

_dimension_rows = []
_dimension_errors = []
for _folder in sorted(p for p in _dimensions_data.iterdir() if p.is_dir()):
    print(f"\nClass: {_folder.name}")
    print(f"{'Image':<55} {'Height':>8} {'Width':>8}")
    for _path in sorted(_folder.rglob("*")):
        if not _path.is_file() or _path.suffix.lower() not in {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}:
            continue
        try:
            with Image.open(_path) as _image:
                # Pillow returns (width, height); apply camera orientation first.
                _width, _height = ImageOps.exif_transpose(_image).size
        except (OSError, ValueError, Image.DecompressionBombError) as _error:
            _dimension_errors.append({"path": str(_path), "error": str(_error)})
            print(f"Unreadable: {_path.name}: {_error}")
            continue
        print(f"{_path.name:<55} {_height:>8} {_width:>8}")
        _dimension_rows.append({"class": _folder.name,
                                "path": _path.relative_to(_dimensions_data).as_posix(),
                                "height": _height, "width": _width})

dimensions = pd.DataFrame(_dimension_rows, columns=["class", "path", "height", "width"])
dimension_summary = dimensions.groupby(
    ["class", "height", "width"], as_index=False
).size().rename(columns={"size": "images"})
print("\nDimension summary (pixels):")
with pd.option_context("display.max_rows", None):
    display(dimension_summary)
print(f"Total: {len(dimensions)} images; unreadable: {len(_dimension_errors)}")


## 3. Check exact duplicates by comparing matrices
`np.array_equal(A, B)` requires identical shapes and every RGB value to match. Compare every pair, including images from different classes. No SHA-256 hash is used here. This pairwise method is suitable for this 500-image sample, but becomes expensive for large datasets.

In [ ]:
pair_columns = ["path_a", "class_a", "path_b", "class_b", "cross_class"]
def pair_record(i, j):
    return {"path_a": paths[i], "class_a": classes[y[i]],
            "path_b": paths[j], "class_b": classes[y[j]],
            "cross_class": bool(y[i] != y[j])}

exact_pairs = []
for i, j in combinations(range(len(image_matrices)), 2):
    if np.array_equal(image_matrices[i], image_matrices[j]):
        exact_pairs.append((i, j))
duplicates = pd.DataFrame([pair_record(i, j) for i, j in exact_pairs], columns=pair_columns)
print("Exact duplicate pairs:", len(duplicates))
display(duplicates.head(5))

### Small matrix-comparison example
If real exact duplicates exist, show the first pair. Otherwise compare the first image with a copy and with a copy containing one changed pixel. These demonstration copies are **not added to the dataset**.

In [ ]:
if exact_pairs:
    i, j = exact_pairs[0]
    example_matrices = [image_matrices[i], image_matrices[j]]
    titles = [paths[i], paths[j]]
    print("Real pair: np.array_equal =", np.array_equal(*example_matrices))
else:
    original = image_matrices[0]
    copied = original.copy()
    modified = original.copy()
    modified[0, 0, 0] = (int(modified[0, 0, 0]) + 1) % 256
    print("No exact duplicates in the dataset. Demonstration only:")
    print("Original vs identical copy:", np.array_equal(original, copied))
    print("Original vs one changed RGB value:", np.array_equal(original, modified))
    print("Original first pixel:", original[0, 0], "Changed first pixel:", modified[0, 0])
    example_matrices = [original, copied, modified]
    titles = ["Original", "Identical copy (demo)", "One changed value (demo)"]
fig, axes = plt.subplots(1, len(example_matrices), figsize=(12, 3), squeeze=False)
for ax, matrix, title in zip(axes[0], example_matrices, titles):
    ax.imshow(matrix)
    ax.set_title(title, fontsize=9)
    ax.axis("off")
show_figure(fig)

## 4. Find near duplicates with dHash
1. Convert each original RGB matrix to grayscale.
2. Resize to **9 columns by 8 rows**.
3. Compare each value with its right neighbor: `1` if right > left, otherwise `0`.
4. Flatten the 8 x 8 comparison matrix into 64 bits.
5. Count differing bits (Hamming distance). Flag pairs with distance **at most 5**.

Exact duplicates are excluded. This checks visual resemblance, not whether two images show the same person. Adjust `THRESHOLD` and rerun from here to explore the candidates.

In [ ]:
THRESHOLD = 5
if not 0 <= THRESHOLD <= 64:
    raise ValueError("THRESHOLD must be between 0 and 64.")
def dhash_steps(matrix):
    grayscale = Image.fromarray(matrix).convert("L")
    small = np.asarray(grayscale.resize((9, 8), Image.Resampling.LANCZOS))
    bits = small[:, 1:] > small[:, :-1]
    value = int.from_bytes(np.packbits(bits.ravel()).tobytes(), "big")
    return small, bits, value

hash_steps = [dhash_steps(matrix) for matrix in image_matrices]
hashes = [step[2] for step in hash_steps]
exact_set = set(exact_pairs)
near_pairs = []
for i, j in combinations(range(len(image_matrices)), 2):
    if (i, j) in exact_set:
        continue
    distance = (hashes[i] ^ hashes[j]).bit_count()
    if distance <= THRESHOLD:
        near_pairs.append((i, j, distance))
near_pairs.sort(key=lambda pair: (pair[2], paths[pair[0]], paths[pair[1]]))
near_duplicates = pd.DataFrame(
    [{**pair_record(i, j), "hamming_distance": distance} for i, j, distance in near_pairs],
    columns=pair_columns + ["hamming_distance"])
print(f"Near-duplicate candidates (distance <= {THRESHOLD}/64): {len(near_pairs)}")
display(near_duplicates.head(5))

## 5. Inspect a small near-duplicate example
Show the first candidate pair, its 9 x 8 intensity matrices, and its 64 comparison bits. Red boxes mark bits that differ. If there are no candidates, show the first image's hashing steps instead.

In [ ]:
if near_pairs:
    i, j, distance = near_pairs[0]
    example_indices = [i, j]
    changed_bits = hash_steps[i][1] != hash_steps[j][1]
    print(f"Candidate distance: {distance} differing bits out of 64. Review visually.")
else:
    example_indices = [0]
    changed_bits = np.zeros((8, 8), dtype=bool)
    print("No near-duplicate candidates. Showing hash steps for the first image.")
fig, axes = plt.subplots(len(example_indices), 4, figsize=(16, 4 * len(example_indices)), squeeze=False)
for row, index in enumerate(example_indices):
    matrix = image_matrices[index]
    small, bits, value = hash_steps[index]
    print("\n", paths[index])
    print("9 x 8 grayscale matrix:\n", small)
    print("8 x 8 comparison matrix:\n", bits.astype(np.uint8))
    print("64-bit dHash:", f"{value:064b}")
    axes[row, 0].imshow(matrix)
    axes[row, 0].set_title(paths[index], fontsize=8)
    axes[row, 1].imshow(Image.fromarray(matrix).convert("L"), cmap="gray", vmin=0, vmax=255)
    axes[row, 1].set_title("Grayscale")
    axes[row, 2].imshow(small, cmap="gray", vmin=0, vmax=255, interpolation="nearest")
    axes[row, 2].set_title("9 columns x 8 rows")
    axes[row, 3].imshow(bits, cmap="gray", vmin=0, vmax=1, interpolation="nearest")
    axes[row, 3].set_title("8 x 8 bits (red = differing)")
    for r in range(8):
        for c in range(9):
            axes[row, 2].text(c, r, str(small[r, c]), ha="center", va="center", fontsize=6,
                              color="black" if small[r, c] > 128 else "white")
        for c in range(8):
            axes[row, 3].text(c, r, str(int(bits[r, c])), ha="center", va="center", fontsize=8,
                              color="black" if bits[r, c] else "white",
                              bbox={"edgecolor": "red", "facecolor": "none", "pad": 2} if changed_bits[r, c] else None)
    for ax in axes[row]:
        ax.axis("off")
show_figure(fig)

## 6. Review results inside the notebook
All results are displayed here. Matrices and tables stay available as Python variables; saving the notebook retains the displayed outputs.


In [ ]:
summary = pd.DataFrame([
    {"metric": "Readable images", "value": len(paths)},
    {"metric": "Classes", "value": len(classes)},
    {"metric": "Exact duplicate pairs", "value": len(exact_pairs)},
    {"metric": "Near-duplicate candidates", "value": len(near_pairs)},
    {"metric": "Unreadable images", "value": len(unreadable)},
])
display(summary)
unreadable_images = pd.DataFrame(unreadable, columns=["path", "error"])
print("Unreadable image details:")
display(unreadable_images)
hash_table = pd.DataFrame({
    "path": paths,
    "class": [classes[label] for label in y],
    "dhash_binary": [f"{value:064b}" for value in hashes],
    "dhash_hex": [f"{value:016x}" for value in hashes],
})
print("Small example of the hash table (full results are in hash_table):")
with pd.option_context("display.max_colwidth", None):
    display(hash_table.head(5))
print("Available variables: X, y, classes, paths, class_counts, dimensions, dimension_summary,")
print("duplicates, near_duplicates, unreadable_images, hash_table, summary")


## 7. Image quality EDA on preprocessed_data
Analyze the manually cleaned dataset at its original resolution. All measurements describe the **whole image**, including background, rather than just the face. No images are altered or automatically removed.

| Metric | Calculation |
| --- | --- |
| Brightness | Mean 8-bit grayscale intensity (0-255) |
| Contrast | Population standard deviation of grayscale intensities |
| Sharpness | Variance of the grayscale Laplacian (`CV_64F`, kernel size 1); lower values are candidates for blur |
| Underexposure | Fraction of grayscale pixels **<= 5** |
| Overexposure | Fraction of grayscale pixels **>= 250** |
| RGB channel means | Mean of each original R, G, B channel |
| Color variation | Population standard deviation of each R, G, B channel |
| Entropy | Shannon entropy `-sum(p * log2(p))` of the 256-bin grayscale histogram (0-8 bits) |
| Edge density | Fraction of nonzero Canny edge pixels, thresholds **100/200**, after a 3 x 3 Gaussian blur (sigma 0) |

Exposure and edge thresholds are configurable in the next cell. Laplacian variance is affected by texture, noise, and image resolution, so the blurriest gallery ranks this measure rather than proving optical blur. Class boxplots include all measurements; gallery categories can overlap.

References: [OpenCV Laplacian](https://docs.opencv.org/4.x/d5/db5/tutorial_laplace_operator.html), [OpenCV Canny](https://docs.opencv.org/4.x/da/d22/tutorial_py_canny.html).

In [ ]:
from pathlib import Path
import io
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from IPython.display import display, Image as DisplayImage

QUALITY_BASE = Path.cwd()
if not (QUALITY_BASE / "preprocessed_data").is_dir():
    QUALITY_BASE = QUALITY_BASE / "analyse"
QUALITY_DATA = QUALITY_BASE / "preprocessed_data"
if not QUALITY_DATA.is_dir():
    raise FileNotFoundError(f"Missing cleaned dataset: {QUALITY_DATA.resolve()}")
QUALITY_RESULTS = QUALITY_DATA / "quality_results"
QUALITY_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
QUALITY_REPORT_FOLDERS = {"mtcnn_results", "quality_results"}
DARK_THRESHOLD, BRIGHT_THRESHOLD = 5, 250
CANNY_LOW, CANNY_HIGH = 100, 200
if not 0 <= DARK_THRESHOLD < BRIGHT_THRESHOLD <= 255:
    raise ValueError("Exposure thresholds must satisfy 0 <= dark < bright <= 255.")
if not 0 <= CANNY_LOW < CANNY_HIGH:
    raise ValueError("Canny thresholds must satisfy 0 <= low < high.")

def quality_show(fig):
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", bbox_inches="tight", dpi=110)
    plt.close(fig)
    display(DisplayImage(data=buffer.getvalue()))

print("Quality input:", QUALITY_DATA.resolve())
print("Exposure thresholds:", DARK_THRESHOLD, BRIGHT_THRESHOLD)
print("Canny thresholds:", CANNY_LOW, CANNY_HIGH)

### Compute the metrics for every cleaned image
RGB is explicitly converted with `COLOR_RGB2GRAY` (Pillow loads RGB). Unreadable images are reported separately. The full results table is available as `quality_metrics`.

In [ ]:
def image_quality_metrics(rgb):
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    histogram = np.bincount(gray.ravel(), minlength=256).astype(np.float64)
    probabilities = histogram[histogram > 0] / gray.size
    channel_means = rgb.mean(axis=(0, 1))
    channel_stds = rgb.std(axis=(0, 1), ddof=0)
    edge_input = cv2.GaussianBlur(gray, (3, 3), 0)
    edges = cv2.Canny(edge_input, CANNY_LOW, CANNY_HIGH)
    return {
        "brightness": float(gray.mean()),
        "contrast": float(gray.std(ddof=0)),
        "sharpness": float(cv2.Laplacian(gray, cv2.CV_64F, ksize=1).var(ddof=0)),
        "underexposure_fraction": float(np.mean(gray <= DARK_THRESHOLD)),
        "overexposure_fraction": float(np.mean(gray >= BRIGHT_THRESHOLD)),
        "mean_r": float(channel_means[0]), "mean_g": float(channel_means[1]),
        "mean_b": float(channel_means[2]),
        "std_r": float(channel_stds[0]), "std_g": float(channel_stds[1]),
        "std_b": float(channel_stds[2]),
        "entropy_bits": float(-np.sum(probabilities * np.log2(probabilities))),
        "edge_density": float(np.count_nonzero(edges) / gray.size),
    }

quality_rows, quality_errors = [], []
for folder in sorted(p for p in QUALITY_DATA.iterdir()
                     if p.is_dir() and p.name not in QUALITY_REPORT_FOLDERS):
    for image_path in sorted(folder.rglob("*")):
        if not image_path.is_file() or image_path.suffix.lower() not in QUALITY_EXTENSIONS:
            continue
        relative = image_path.relative_to(QUALITY_DATA).as_posix()
        try:
            with Image.open(image_path) as source:
                rgb = np.array(ImageOps.exif_transpose(source).convert("RGB"))
            quality_rows.append({"class": folder.name, "path": relative,
                                 "height": rgb.shape[0], "width": rgb.shape[1],
                                 **image_quality_metrics(rgb)})
        except (OSError, ValueError, cv2.error, Image.DecompressionBombError) as error:
            quality_errors.append({"class": folder.name, "path": relative, "error": str(error)})
if not quality_rows:
    raise ValueError("No readable images in preprocessed_data.")
quality_metrics = pd.DataFrame(quality_rows).sort_values(["class", "path"]).reset_index(drop=True)
quality_unreadable = pd.DataFrame(quality_errors, columns=["class", "path", "error"])
quality_class_counts = quality_metrics.groupby("class").size().rename("images").reset_index()
print(f"Measured {len(quality_metrics)} images in {len(quality_class_counts)} classes.")
print("Unreadable images:", len(quality_unreadable))
display(quality_class_counts)
with pd.option_context("display.max_columns", None):
    display(quality_metrics.head(5))
    display(quality_metrics.select_dtypes(include=np.number).describe().T)
if not quality_unreadable.empty:
    display(quality_unreadable)

### Histograms and class boxplots
The top row shows overall distributions; the bottom row compares classes for brightness, contrast, and sharpness. Boxplot points beyond whiskers are statistical outliers, not automatic removal decisions.

In [ ]:
quality_plot_columns = ["brightness", "contrast", "sharpness"]
quality_axis_labels = ["Mean grayscale intensity", "Grayscale standard deviation", "Laplacian variance"]
quality_class_names = quality_class_counts["class"].tolist()
fig, axes = plt.subplots(2, 3, figsize=(19, 11))
for column, metric, label in zip(range(3), quality_plot_columns, quality_axis_labels):
    axes[0, column].hist(quality_metrics[metric], bins=30, color="#3677ba", edgecolor="white")
    axes[0, column].set_title(f"{metric.title()} histogram")
    axes[0, column].set_xlabel(label)
    axes[0, column].set_ylabel("Images")
    grouped_values = [quality_metrics.loc[quality_metrics["class"] == name, metric].to_numpy()
                      for name in quality_class_names]
    axes[1, column].boxplot(grouped_values, tick_labels=quality_class_names, patch_artist=True,
                           boxprops={"facecolor": "#9fc4e8"}, medianprops={"color": "#a22d22"})
    axes[1, column].set_title(f"{metric.title()} by class")
    axes[1, column].set_ylabel(label)
    axes[1, column].tick_params(axis="x", labelrotation=65, labelsize=8)
    axes[1, column].grid(axis="y", alpha=0.2)
fig.tight_layout()
quality_show(fig)

### Brightness versus contrast and sharpness
Each point is an image, colored by its class. These plots describe associations, not causal effects.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
colors = plt.get_cmap("tab10")
for index, class_name in enumerate(quality_class_names):
    group = quality_metrics.loc[quality_metrics["class"] == class_name]
    for ax, metric in zip(axes, ["contrast", "sharpness"]):
        ax.scatter(group["brightness"], group[metric], color=colors(index % 10),
                   label=class_name, alpha=0.65, s=24)
for ax, metric, label in zip(axes, ["contrast", "sharpness"], quality_axis_labels[1:]):
    ax.set_xlabel("Brightness (mean grayscale intensity)")
    ax.set_ylabel(label)
    ax.set_title(f"Brightness versus {metric}")
    ax.grid(alpha=0.2)
axes[1].legend(loc="upper left", bbox_to_anchor=(1.02, 1), fontsize=8)
fig.tight_layout()
quality_show(fig)
print("Pearson correlations (descriptive):")
display(quality_metrics[quality_plot_columns].corr())

### Five darkest, brightest, blurriest, and lowest-contrast images
Rank images by the specified metric. Blurriest means the five lowest Laplacian variances. Every thumbnail includes the class, filename, and score. Original colors and intensity ranges are preserved. Images can appear in multiple categories.

In [ ]:
quality_rankings = [
    ("Five darkest", "brightness", True),
    ("Five brightest", "brightness", False),
    ("Five blurriest (lowest Laplacian variance)", "sharpness", True),
    ("Five lowest-contrast", "contrast", True),
]
quality_extreme_rows = []
for title, metric, ascending in quality_rankings:
    selected = quality_metrics.sort_values([metric, "path"], ascending=[ascending, True]).head(5)
    print(title)
    display(selected[["class", "path", "brightness", "contrast", "sharpness"]].reset_index(drop=True))
    fig, axes = plt.subplots(1, len(selected), figsize=(4 * len(selected), 4), squeeze=False)
    fig.suptitle(title, fontsize=14)
    for rank, (ax, (_, row)) in enumerate(zip(axes[0], selected.iterrows()), start=1):
        with Image.open(QUALITY_DATA / row["path"]) as source:
            ax.imshow(ImageOps.exif_transpose(source).convert("RGB"))
        ax.set_title(f"{row['class']}\n{Path(row['path']).name}\n{metric} = {row[metric]:.2f}", fontsize=9)
        ax.axis("off")
        quality_extreme_rows.append({"category": title, "rank": rank, **row.to_dict()})
    fig.tight_layout()
    quality_show(fig)
quality_extremes = pd.DataFrame(quality_extreme_rows)

### Save metrics beside the cleaned dataset
All tables and plots are displayed inline above. CSV measurements, selected extremes, and calculation settings are also saved under `preprocessed_data/quality_results`; class images stay intact.

In [ ]:
import json
QUALITY_RESULTS.mkdir(parents=True, exist_ok=True)
quality_metrics.to_csv(QUALITY_RESULTS / "image_quality.csv", index=False)
quality_metrics.groupby("class")[quality_plot_columns].agg(["count", "mean", "std", "min", "median", "max"]).to_csv(
    QUALITY_RESULTS / "class_summary.csv")
quality_unreadable.to_csv(QUALITY_RESULTS / "unreadable.csv", index=False)
quality_extremes.to_csv(QUALITY_RESULTS / "extreme_images.csv", index=False)
quality_settings = {
    "dataset": str(QUALITY_DATA.resolve()), "scope": "whole image, EXIF orientation corrected, original resolution",
    "grayscale": "cv2.COLOR_RGB2GRAY, uint8", "std_ddof": 0,
    "dark_threshold_inclusive": DARK_THRESHOLD, "bright_threshold_inclusive": BRIGHT_THRESHOLD,
    "laplacian_depth": "CV_64F", "laplacian_ksize": 1,
    "edge_gaussian_kernel": [3, 3], "edge_gaussian_sigma": 0,
    "canny_thresholds": [CANNY_LOW, CANNY_HIGH], "entropy_bins": 256, "entropy_log_base": 2,
    "opencv_version": cv2.__version__, "readable_images": len(quality_metrics),
}
(QUALITY_RESULTS / "settings.json").write_text(json.dumps(quality_settings, indent=2) + "\n", encoding="utf-8")
print("Saved image-quality measurements to", QUALITY_RESULTS.resolve())

## 8. MTCNN face detection on preprocessed_data
This section runs independently of the earlier EDA cells. Read the manually cleaned class folders in `analyse/preprocessed_data`. Images are left intact. Save measurements in `preprocessed_data/mtcnn_results`.

- **Confidence:** MTCNN's detection score, not a calibrated probability of correct identity.
- **Box:** raw detector `(x, y, width, height)` plus the box clipped to the image boundaries. Area ratio uses the clipped box area divided by image area, not a segmented face area.
- **Position:** clipped box center in pixels and normalized coordinates (0 = left/top, 1 = right/bottom), plus offset from the image center.
- **Eye angle:** eye line in degrees from horizontal, with eyes ordered by screen x coordinate. Positive means the line slopes downward to the right; missing or coincident eyes produce NaN. This measures in-plane rotation, not head yaw or pitch.
- **Landmarks:** availability and image-boundary validity of both eyes, nose, and both mouth corners. Availability does not establish accuracy.

All returned detections are retained. Images with no face or read/detection errors are explicitly recorded. The highest-confidence detection is marked as the primary face; it is not guaranteed to be the labeled person.

API reference: https://mtcnn.readthedocs.io/en/latest/usage_params/

In [ ]:
from pathlib import Path
import math
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from IPython.display import display

MTCNN_BASE = Path.cwd()
if not (MTCNN_BASE / "preprocessed_data").is_dir():
    MTCNN_BASE = MTCNN_BASE / "analyse"
MTCNN_DATA = MTCNN_BASE / "preprocessed_data"
if not MTCNN_DATA.is_dir():
    raise FileNotFoundError(f"Missing dataset: {MTCNN_DATA.resolve()}")
MTCNN_RESULTS = MTCNN_DATA / "mtcnn_results"
MTCNN_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
LANDMARKS = ("left_eye", "right_eye", "nose", "mouth_left", "mouth_right")
print("Input:", MTCNN_DATA.resolve())
print("Results:", MTCNN_RESULTS.resolve())

### Measure each detection
The following helper contains the measurement logic and can be run without TensorFlow.

In [ ]:
def measure_face(face, image_width, image_height):
    x, y, width, height = map(float, face["box"])
    x1 = float(np.clip(x, 0, image_width))
    y1 = float(np.clip(y, 0, image_height))
    x2 = float(np.clip(x + width, 0, image_width))
    y2 = float(np.clip(y + height, 0, image_height))
    clipped_width, clipped_height = max(0.0, x2 - x1), max(0.0, y2 - y1)
    valid_box = clipped_width > 0 and clipped_height > 0
    cx, cy = ((x1 + x2) / 2, (y1 + y2) / 2) if valid_box else (np.nan, np.nan)
    result = {
        "confidence": float(face["confidence"]),
        "bbox_x": x, "bbox_y": y, "bbox_width": width, "bbox_height": height,
        "bbox_area": max(0.0, width) * max(0.0, height),
        "clipped_x": x1, "clipped_y": y1,
        "clipped_width": clipped_width, "clipped_height": clipped_height,
        "valid_box": valid_box,
        "box_was_clipped": (x1, y1, x2, y2) != (x, y, x + width, y + height),
        "face_to_image_area_ratio": clipped_width * clipped_height / (image_width * image_height),
        "face_center_x": cx, "face_center_y": cy,
        "face_center_x_normalized": cx / image_width,
        "face_center_y_normalized": cy / image_height,
        "face_offset_x_normalized": cx / image_width - 0.5,
        "face_offset_y_normalized": cy / image_height - 0.5,
    }
    points = face.get("keypoints") or {}
    available_points = {}
    for name in LANDMARKS:
        point = points.get(name)
        available = point is not None and len(point) == 2 and bool(np.isfinite(point).all())
        px, py = map(float, point) if available else (np.nan, np.nan)
        in_image = available and 0 <= px < image_width and 0 <= py < image_height
        result.update({f"{name}_available": available, f"{name}_in_image": bool(in_image),
                       f"{name}_x": px, f"{name}_y": py})
        if available:
            available_points[name] = (px, py)
    result["landmarks_available_count"] = len(available_points)
    result["all_landmarks_available"] = len(available_points) == len(LANDMARKS)
    result["all_landmarks_in_image"] = all(result[f"{name}_in_image"] for name in LANDMARKS)
    result["eye_angle_degrees"] = np.nan
    if "left_eye" in available_points and "right_eye" in available_points:
        left, right = sorted([available_points["left_eye"], available_points["right_eye"]])
        dx, dy = right[0] - left[0], right[1] - left[1]
        if math.hypot(dx, dy) > 0:
            result["eye_angle_degrees"] = math.degrees(math.atan2(dy, dx))
    result["eye_angle_available"] = bool(np.isfinite(result["eye_angle_degrees"]))
    result["eye_angle_abs_degrees"] = abs(result["eye_angle_degrees"])
    return result

### Run the detector and save measurements
This cell requires a working TensorFlow/MTCNN installation. Initialization and detection failures are reported rather than replaced with invented results.

In [ ]:
# Initialize before writing results, so a runtime failure cannot replace valid reports.
try:
    from mtcnn import MTCNN
    mtcnn_detector = MTCNN()
except Exception as error:
    raise RuntimeError(
        "MTCNN could not initialize. If TensorFlow reports an Application Control/DLL block, "
        "use an approved environment or contact IT. No detection reports were written."
    ) from error

mtcnn_files = []
for folder in sorted(MTCNN_DATA.iterdir()):
    if folder.is_dir() and folder.name not in {"mtcnn_results", "quality_results"}:
        mtcnn_files.extend((folder.name, p) for p in sorted(folder.rglob("*"))
                           if p.is_file() and p.suffix.lower() in MTCNN_EXTENSIONS)
if not mtcnn_files:
    raise ValueError("No images found in preprocessed_data class folders.")

mtcnn_detection_rows, mtcnn_image_rows = [], []
for index, (class_name, image_path) in enumerate(mtcnn_files, start=1):
    relative_path = image_path.relative_to(MTCNN_DATA).as_posix()
    status = {"path": relative_path, "class": class_name,
              "image_width": None, "image_height": None,
              "status": "ok", "detections": 0, "max_confidence": np.nan, "error": ""}
    try:
        with Image.open(image_path) as source:
            rgb = np.array(ImageOps.exif_transpose(source).convert("RGB"))
        height, width = rgb.shape[:2]
        status.update(image_width=width, image_height=height)
    except Exception as error:
        status.update(status="read_error", error=str(error))
        mtcnn_image_rows.append(status)
        continue
    try:
        faces = mtcnn_detector.detect_faces(rgb)
        primary = max(range(len(faces)), key=lambda i: faces[i]["confidence"]) if faces else None
        # Build all rows before committing this image's results.
        image_rows = [
            {"path": relative_path, "class": class_name, "detection_id": face_id,
             "is_primary_face": face_id == primary,
             "image_width": width, "image_height": height,
             **measure_face(face, width, height)}
            for face_id, face in enumerate(faces)
        ]
        mtcnn_detection_rows.extend(image_rows)
        status.update(status="ok" if faces else "no_face", detections=len(faces),
                      max_confidence=max((float(f["confidence"]) for f in faces), default=np.nan))
    except Exception as error:
        status.update(status="detection_error", error=str(error))
    mtcnn_image_rows.append(status)
    if index % 25 == 0 or index == len(mtcnn_files):
        print(f"Processed {index}/{len(mtcnn_files)} images", flush=True)

measurement_columns = list(measure_face(
    {"box": [0, 0, 1, 1], "confidence": 0.0, "keypoints": {}}, 1, 1))
mtcnn_detections = pd.DataFrame(mtcnn_detection_rows, columns=[
    "path", "class", "detection_id", "is_primary_face", "image_width", "image_height",
    *measurement_columns])
mtcnn_images = pd.DataFrame(mtcnn_image_rows)
mtcnn_class_summary = mtcnn_images.groupby("class", as_index=False).agg(
    images=("path", "size"), faces_detected=("detections", "sum"),
    images_with_faces=("detections", lambda s: int((s > 0).sum())),
    images_with_multiple_faces=("detections", lambda s: int((s > 1).sum())),
    no_face_images=("status", lambda s: int((s == "no_face").sum())),
    failed_images=("status", lambda s: int(s.isin(["read_error", "detection_error"]).sum())))
MTCNN_RESULTS.mkdir(parents=True, exist_ok=True)
for filename, table in [("detections.csv", mtcnn_detections),
                        ("image_status.csv", mtcnn_images),
                        ("class_summary.csv", mtcnn_class_summary)]:
    table.to_csv(MTCNN_RESULTS / filename, index=False)
print("Saved measurements to", MTCNN_RESULTS.resolve())
display(mtcnn_class_summary)
display(mtcnn_detections.head(5))
display(mtcnn_images.loc[mtcnn_images["status"] != "ok"].head(10))

### Small visual examples
Show up to three images with all detected boxes and available landmarks. The annotation is displayed inside the notebook; originals in preprocessed_data remain unchanged. Detection measurements are saved beside the dataset in mtcnn_results.

In [ ]:
import io
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from IPython.display import Image as DisplayImage

example_paths = mtcnn_detections["path"].drop_duplicates().head(3).tolist()
if not example_paths:
    print("No detected faces to display. Inspect mtcnn_images for no-face images or errors.")
else:
    fig, axes = plt.subplots(1, len(example_paths), figsize=(6 * len(example_paths), 6), squeeze=False)
    for ax, relative_path in zip(axes[0], example_paths):
        with Image.open(MTCNN_DATA / relative_path) as source:
            ax.imshow(ImageOps.exif_transpose(source).convert("RGB"))
        for _, face in mtcnn_detections.loc[mtcnn_detections["path"] == relative_path].iterrows():
            ax.add_patch(Rectangle((face["clipped_x"], face["clipped_y"]),
                                   face["clipped_width"], face["clipped_height"],
                                   fill=False, edgecolor="lime", linewidth=2))
            angle = f"{face['eye_angle_degrees']:.1f} deg" if face["eye_angle_available"] else "missing eyes"
            ax.text(face["clipped_x"], face["clipped_y"],
                    f"conf={face['confidence']:.3f} | angle={angle}\narea ratio={face['face_to_image_area_ratio']:.3f}",
                    fontsize=8, color="white", bbox={"facecolor": "black", "alpha": 0.6})
            for name in LANDMARKS:
                if face[f"{name}_available"]:
                    ax.scatter(face[f"{name}_x"], face[f"{name}_y"], s=18, color="yellow")
        ax.set_title(relative_path, fontsize=9)
        ax.axis("off")
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", bbox_inches="tight", dpi=110)
    plt.close(fig)
    display(DisplayImage(data=buffer.getvalue()))

# 9. Interpret the quality reports
Reload the saved CSV reports without rerunning MTCNN. Statistics and plots are calculated from those files. Whole-image brightness includes the background; a dark background does not necessarily mean an underexposed face. Quality thresholds flag images for inspection.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display
ROOT = Path.cwd()
if not (ROOT / "preprocessed_data").is_dir():
    ROOT = ROOT / "analyse"
DATA = ROOT / "preprocessed_data"
QDIR, DDIR = DATA / "quality_results", DATA / "mtcnn_results"
# Also accept reports moved beside the notebook.
def report_file(folder, name):
    for candidate in (folder/name, ROOT/folder.name/name, ROOT/name, Path.cwd()/name):
        if candidate.exists(): return candidate
    raise FileNotFoundError(f"{name} not found; run the report-generation cells.")
q = pd.read_csv(report_file(QDIR, "image_quality.csv"))
det = pd.read_csv(report_file(DDIR, "detections.csv"))
status = pd.read_csv(report_file(DDIR, "image_status.csv"))
print(f"{len(q)} readable images, {q['class'].nunique()} identities, {len(det)} detections")
display(q.groupby("class").agg(images=("path","count"), brightness=("brightness","mean"),
                               contrast=("contrast","mean"), sharpness=("sharpness","mean")).round(2))
print("Images with multiple detections:", int((status.detections>1).sum()))
print("Images without faces:", int((status.detections==0).sum()))


## 9.1 Compare identities and distributions
An identity with darker or less sharp photos may be harder to classify. Check the recognition scores for each identity before drawing that conclusion.

In [ ]:
metrics = {"brightness":"Mean brightness", "contrast":"Mean contrast", "sharpness":"Mean sharpness (Laplacian variance)"}
grouped = q.groupby("class")[list(metrics)].mean()
fig, axes = plt.subplots(1, 3, figsize=(19, 6))
for ax,(col,title) in zip(axes,metrics.items()):
    grouped[col].sort_values().plot.barh(ax=ax, color="#2386C8")
    ax.set_title(title); ax.set_xlabel(col); ax.set_ylabel("")
fig.tight_layout(); plt.show()
fig, axes = plt.subplots(1,3,figsize=(17,4))
for ax,(col,title) in zip(axes,metrics.items()):
    ax.hist(q[col].dropna(), bins=20, edgecolor="white")
    ax.axvline(q[col].median(), color="darkorange", ls="--", label="Median")
    ax.set_title(f"Distribution : {title}"); ax.legend()
fig.tight_layout(); plt.show()
display(q[list(metrics)].describe().round(2).T)


## 9.2 Correlations and quality outliers
- Contrast and Laplacian variance share a dependence on intensity changes.
- Do not apply gamma correction based only on whole-image brightness.
- High sharpness is usually desirable. Extreme values can also reflect noise or compression.
- IQR outliers are candidates for inspection, not confirmed errors.

In [ ]:
cols = ["brightness","contrast","sharpness","edge_density","entropy_bits"]
corr = q[cols].corr()
fig, ax = plt.subplots(figsize=(7,6))
im=ax.imshow(corr, cmap="coolwarm", vmin=-1,vmax=1)
ax.set_xticks(range(len(cols)),cols,rotation=40,ha="right")
ax.set_yticks(range(len(cols)),cols)
for i in range(len(cols)):
    for j in range(len(cols)):
        ax.text(j,i,f"{corr.iloc[i,j]:.2f}",ha="center",va="center")
fig.colorbar(im,ax=ax); ax.set_title("Pearson correlations"); fig.tight_layout(); plt.show()
flags = q[["class","path","brightness","contrast","sharpness"]].copy()
for col in ("brightness","contrast","sharpness"):
    a,b=q[col].quantile([.25,.75]); spread=b-a
    flags[f"{col}_iqr_flag"] = (q[col]<a-1.5*spread)|(q[col]>b+1.5*spread)
print("Outlier candidates by variable :")
display(flags[[c for c in flags if c.endswith("_flag")]].sum().to_frame("images"))
display(flags.loc[flags.filter(like="_flag").any(axis=1)].head(30))


# 10. Select one primary face
Compare highest confidence, largest box, and a combination of relative size and centrality. The combined score is 60% relative box area, 35% centrality, and 5% detection confidence. Relative area is normalized within each image. This heuristic favors a large central face, but does not verify identity. Review ambiguous cases before training.

In [ ]:
def choose_primary_faces(detections, method="center_size", center_weight=0.35):
    if method not in {"center_size","largest","confidence"}:
        raise ValueError("method must be center_size, largest or confidence")
    d=detections.loc[detections["valid_box"].astype(str).str.lower().eq("true")].copy()
    if d.empty:
        return d.assign(selection_score=pd.Series(dtype=float))
    area=d["face_to_image_area_ratio"].fillna(0).clip(lower=0)
    cx=d["face_center_x_normalized"].fillna(-1)
    cy=d["face_center_y_normalized"].fillna(-1)
    distance=np.sqrt((cx-.5)**2+(cy-.5)**2)
    centrality=(1-distance/np.sqrt(.5)).clip(0,1)
    if method=="center_size":
        # Normalize area within each image.
        max_area=area.groupby(d["path"]).transform("max").replace(0,np.nan)
        relative_area=(area/max_area).fillna(0)
        score=0.60*relative_area+center_weight*centrality+0.05*d["confidence"].fillna(0)
    elif method=="largest": score=area
    else: score=d["confidence"].fillna(0)
    score=score.where(d["valid_box"].fillna(False).astype(bool),-np.inf)
    d["selection_score"]=score
    chosen=d.loc[d.groupby("path")["selection_score"].idxmax()].copy()
    return chosen.sort_values("path").reset_index(drop=True)

selected = choose_primary_faces(det, "center_size")
baseline = det.loc[det["is_primary_face"].astype(str).str.lower().eq("true")]
comparison=selected[["path","detection_id"]].merge(
    baseline[["path","detection_id"]],on="path",suffixes=("_center_size","_confidence"))
changed=comparison[comparison.detection_id_center_size!=comparison.detection_id_confidence]
print(f"Selections differing from confidence alone : {len(changed)} / {len(comparison)} images")
display(changed.head(30))

## 10.1 Review ambiguous selections
The selected box is green; other detections are orange. Inspect images where selection rules disagree, images with multiple detections, and unusually small boxes.

In [ ]:
from PIL import Image, ImageOps
from matplotlib.patches import Rectangle
review_paths = list(dict.fromkeys(changed.path.tolist() + status.loc[status.detections>1,"path"].tolist()))
def show_face_choices(paths_to_show, max_images=18):
    sample=paths_to_show[:max_images]
    if not sample: print("No images to inspect"); return
    fig,axs=plt.subplots((len(sample)+3)//4,4,figsize=(16,4*((len(sample)+3)//4)))
    axs=np.asarray(axs).reshape(-1)
    choice=dict(zip(selected.path,selected.detection_id))
    for ax,rel in zip(axs,sample):
        f=DATA/rel
        if not f.exists():
            ax.text(.5,.5,f"Introuvable:\n{rel}",ha="center");ax.axis("off");continue
        with Image.open(f) as src: ax.imshow(ImageOps.exif_transpose(src).convert("RGB"))
        for _,r in det.loc[det.path==rel].iterrows():
            picked=int(r.detection_id)==int(choice[rel])
            ax.add_patch(Rectangle((r.clipped_x,r.clipped_y),r.clipped_width,r.clipped_height,
                                   fill=False,edgecolor="lime" if picked else "orange",linewidth=2))
        ax.set_title(rel.split("/")[-1],fontsize=9);ax.axis("off")
    for ax in axs[len(sample):]:ax.axis("off")
    plt.tight_layout();plt.show()
show_face_choices(review_paths)


# 11. Filter preprocessed_data and create MTCNN crops
Recompute sharpness at original resolution. Back up images with Laplacian variance **strictly below 20**, verify their copies, and remove them from `preprocessed_data`. Keep sharp images. The threshold is configurable and does not measure recognition accuracy.

Rejected files go to `analyse/rejected_low_sharpness`. Retained files keep their original class folders and encoding. `analyse/data` stays intact. Earlier reports describe the data before filtering; this section writes separate filtering reports.

In [ ]:
from pathlib import Path
import json, shutil
import cv2
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from IPython.display import display, Markdown

MIN_SHARPNESS = 20.0
assert np.isfinite(MIN_SHARPNESS) and MIN_SHARPNESS >= 0
DATA = DATA.resolve()
assert DATA.name == "preprocessed_data" and DATA.is_dir()
BACKUP_DIR = DATA.parent / "rejected_low_sharpness"
FILTER_RESULTS = DATA / "quality_results" / "filtering"
FILTER_RESULTS.mkdir(parents=True, exist_ok=True)
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
REPORT_DIRS = {"quality_results", "mtcnn_results"}

def dataset_images(root):
    for folder in sorted(root.iterdir()):
        if folder.is_dir() and folder.name not in REPORT_DIRS:
            for image_path in sorted(folder.rglob("*")):
                if image_path.is_file() and image_path.suffix.lower() in IMAGE_EXTENSIONS:
                    yield folder.name, image_path

def simple_metrics(rgb):
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    return {"width": rgb.shape[1], "height": rgb.shape[0],
            "brightness": float(gray.mean()), "contrast": float(gray.std()),
            "sharpness": float(cv2.Laplacian(gray, cv2.CV_64F, ksize=1).var())}

filter_rows = []
for class_name, src in dataset_images(DATA):
    src = src.resolve()
    assert src.is_relative_to(DATA), f"File outside dataset : {src}"
    rel = src.relative_to(DATA)
    with Image.open(src) as image:
        rgb = np.asarray(ImageOps.exif_transpose(image).convert("RGB"))
    filter_rows.append({"class": class_name, "path": rel.as_posix(), **simple_metrics(rgb)})
filter_audit = pd.DataFrame(filter_rows)
if filter_audit.empty:
    raise ValueError("No images to filter.")
filter_audit["removed"] = filter_audit.sharpness < MIN_SHARPNESS
filter_audit.to_csv(FILTER_RESULTS / "filter_audit.csv", index=False)

# Back up and verify each file before removing it from the dataset.
import hashlib
removed_now = 0
for row in filter_audit.loc[filter_audit.removed].itertuples():
    src = (DATA / row.path).resolve()
    dst = (BACKUP_DIR / row.path).resolve()
    assert src.is_relative_to(DATA) and dst.is_relative_to(BACKUP_DIR.resolve())
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() and dst.read_bytes() != src.read_bytes():
        raise FileExistsError(f"Backup differs : {dst}")
    shutil.copy2(src, dst)
    assert hashlib.sha256(src.read_bytes()).digest() == hashlib.sha256(dst.read_bytes()).digest()
    src.unlink()
    removed_now += 1

clean_quality = filter_audit.loc[~filter_audit.removed].drop(columns="removed").copy()
clean_quality.to_csv(FILTER_RESULTS / "retained_image_quality.csv", index=False)
filter_counts = filter_audit.groupby("class").agg(
    before=("path", "size"), removed=("removed", "sum"))
filter_counts["after"] = filter_counts.before - filter_counts.removed
filter_counts.to_csv(FILTER_RESULTS / "class_counts.csv")
(FILTER_RESULTS / "settings.json").write_text(json.dumps({
    "min_sharpness": MIN_SHARPNESS, "comparison": "sharpness < min_sharpness",
    "metric": "variance of cv2.Laplacian(COLOR_RGB2GRAY, CV_64F, ksize=1)",
    "backup": str(BACKUP_DIR), "removed_this_run": removed_now,
    "retained": len(clean_quality)}, indent=2), encoding="utf-8")
print(f"Threshold: {MIN_SHARPNESS}; removed: {removed_now}; retained: {len(clean_quality)}")
display(filter_counts)
display(filter_audit.loc[filter_audit.removed, ["class", "path", "sharpness"]])

## 11.1 Export one selected face per image
Rerun MTCNN on retained sources and use the same `choose_primary_faces(..., "center_size")` rule as section 10. Export one valid box per image, retaining the source identity label. Report sources without valid detections.

Save RGB PNG crops at **160 x 160** in `analyse/cropped_faces/<class>/`. This step crops and resizes; eye alignment happens later in the model-specific scripts. A selected box may still overlap another face. Save its original coordinates and dimensions. On reruns, replace only crops listed in the previous manifest.

In [ ]:
from mtcnn import MTCNN
import matplotlib.pyplot as plt

CROP_DIR = DATA.parent / "cropped_faces"
CROP_REPORTS = CROP_DIR / "reports"
CROP_SIZE = (160, 160)
CROP_REPORTS.mkdir(parents=True, exist_ok=True)
detector = MTCNN()
clean_files = list(dataset_images(DATA))
fresh_detections, fresh_status = [], []
for index, (class_name, src) in enumerate(clean_files, 1):
    rel = src.relative_to(DATA).as_posix()
    with Image.open(src) as image:
        rgb = np.asarray(ImageOps.exif_transpose(image).convert("RGB"))
    height, width = rgb.shape[:2]
    # Stop export on a detector error.
    faces = detector.detect_faces(rgb)
    valid_count = 0
    for face_id, face in enumerate(faces):
        x, y, w, h = map(float, face["box"])
        x1, y1 = max(0, x), max(0, y)
        x2, y2 = min(width, x + w), min(height, y + h)
        valid = bool(np.isfinite([x1, y1, x2, y2]).all() and x2 > x1 and y2 > y1)
        valid_count += int(valid)
        fresh_detections.append({"path": rel, "class": class_name,
            "detection_id": face_id, "confidence": float(face["confidence"]),
            "valid_box": valid, "clipped_x": x1, "clipped_y": y1,
            "clipped_width": max(0, x2-x1), "clipped_height": max(0, y2-y1),
            "face_to_image_area_ratio": max(0, x2-x1)*max(0, y2-y1)/(width*height),
            "face_center_x_normalized": (x1+x2)/(2*width),
            "face_center_y_normalized": (y1+y2)/(2*height)})
    fresh_status.append({"class": class_name, "path": rel, "detections": len(faces),
                         "valid_detections": valid_count,
                         "status": "ok" if valid_count else "no_valid_face"})
    if index % 50 == 0 or index == len(clean_files):
        print(f"MTCNN : {index}/{len(clean_files)}", flush=True)

face_columns = ["path", "class", "detection_id", "confidence", "valid_box",
    "clipped_x", "clipped_y", "clipped_width", "clipped_height",
    "face_to_image_area_ratio", "face_center_x_normalized", "face_center_y_normalized"]
fresh_det = pd.DataFrame(fresh_detections, columns=face_columns)
crop_status = pd.DataFrame(fresh_status)
crop_selected = choose_primary_faces(fresh_det, "center_size")
assert crop_selected.path.is_unique
if crop_selected.empty:
    raise ValueError("No valid face: check MTCNN before creating the dataset.")

manifest_path = CROP_REPORTS / "crop_manifest.csv"
previous = pd.read_csv(manifest_path) if manifest_path.exists() else pd.DataFrame()
manifest = []
for face in crop_selected.itertuples():
    with Image.open(DATA / face.path) as source:
        image = ImageOps.exif_transpose(source).convert("RGB")
        box = (max(0, int(np.floor(face.clipped_x))), max(0, int(np.floor(face.clipped_y))),
               min(image.width, int(np.ceil(face.clipped_x + face.clipped_width))),
               min(image.height, int(np.ceil(face.clipped_y + face.clipped_height))))
        crop = image.crop(box)
        original_width, original_height = crop.size
        # Preserve the source extension to avoid filename collisions.
        crop_rel = Path(face.path + ".png")
        dst = (CROP_DIR / crop_rel).resolve()
        assert dst.is_relative_to(CROP_DIR.resolve())
        dst.parent.mkdir(parents=True, exist_ok=True)
        crop.resize(CROP_SIZE, Image.Resampling.LANCZOS).save(dst)
    manifest.append({"class": face._asdict()["class"] if "class" in face._asdict() else Path(face.path).parts[0],
        "source": face.path, "crop": crop_rel.as_posix(), "detection_id": face.detection_id,
        "selection_score": face.selection_score, "confidence": face.confidence,
        "x1": box[0], "y1": box[1], "x2": box[2], "y2": box[3],
        "original_crop_width": original_width, "original_crop_height": original_height})
crop_manifest = pd.DataFrame(manifest)
assert crop_manifest.source.is_unique and crop_manifest.crop.is_unique
if not previous.empty:
    for old_rel in set(previous.crop) - set(crop_manifest.crop):
        old_path = (CROP_DIR / old_rel).resolve()
        assert old_path.is_relative_to(CROP_DIR.resolve()) and old_path.suffix == ".png"
        if old_path.is_file():
            old_path.unlink()
crop_manifest.to_csv(manifest_path, index=False)
fresh_det.to_csv(CROP_REPORTS / "detections.csv", index=False)
crop_selected.to_csv(CROP_REPORTS / "selected_faces.csv", index=False)
crop_status.to_csv(CROP_REPORTS / "image_status.csv", index=False)
print(f"{len(crop_manifest)} crops saved in {CROP_DIR.resolve()}")
print("Images without valid faces :", int((crop_status.status != "ok").sum()))
display(crop_status.loc[crop_status.status != "ok"])
display(crop_manifest.head())

## 11.2 Analyze the saved crops
Calculate class counts, dimensions, brightness, contrast, and sharpness on the exported PNG files. Compare each crop with its own source. Resizing changes Laplacian variance, so a higher value does not prove better recognition. Do not automatically reapply the original-image threshold to crops.

In [ ]:
# Run this analysis independently of filtering and detection.
from pathlib import Path
import io
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from IPython.display import display, Markdown, Image as DisplayImage

ROOT = Path.cwd()
if not (ROOT / "preprocessed_data").is_dir():
    ROOT = ROOT / "analyse"
DATA = ROOT / "preprocessed_data"
CROP_DIR = ROOT / "cropped_faces"
CROP_REPORTS = CROP_DIR / "reports"
FILTER_RESULTS = DATA / "quality_results" / "filtering"
CHART_DIR = CROP_REPORTS / "graphs"
CHART_DIR.mkdir(parents=True, exist_ok=True)
crop_manifest = pd.read_csv(CROP_REPORTS / "crop_manifest.csv")
crop_status = pd.read_csv(CROP_REPORTS / "image_status.csv")
filter_audit = pd.read_csv(FILTER_RESULTS / "filter_audit.csv")
clean_quality = pd.read_csv(FILTER_RESULTS / "retained_image_quality.csv")
filter_counts = pd.read_csv(FILTER_RESULTS / "class_counts.csv", index_col="class")
removed_now = int(filter_audit.removed.astype(str).str.lower().eq("true").sum())

def simple_metrics(rgb):
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    return {"width": rgb.shape[1], "height": rgb.shape[0],
            "brightness": float(gray.mean()), "contrast": float(gray.std()),
            "sharpness": float(cv2.Laplacian(gray, cv2.CV_64F, ksize=1).var())}

def quality_show(fig, filename=None):
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", bbox_inches="tight", dpi=120)
    if filename:
        (CHART_DIR / filename).write_bytes(buffer.getvalue())
    plt.close(fig)
    display(DisplayImage(data=buffer.getvalue()))

display(Markdown("### Results from the saved crops"))

cropped_rows = []
for row in crop_manifest.itertuples():
    with Image.open(CROP_DIR / row.crop) as image:
        rgb = np.asarray(image.convert("RGB"))
    cropped_rows.append({"class": Path(row.source).parts[0], "source": row.source,
                         "crop": row.crop, **simple_metrics(rgb)})
cropped_quality = pd.DataFrame(cropped_rows)
cropped_quality.to_csv(CROP_REPORTS / "image_quality.csv", index=False)
crop_counts = cropped_quality.groupby("class").size().rename("crops")
count_comparison = filter_counts.join(crop_counts).fillna({"crops": 0}).astype(int)
count_comparison["without_crop"] = count_comparison.after - count_comparison.crops
display(count_comparison)
fig, ax = plt.subplots(figsize=(11, 4))
crop_counts.plot.bar(ax=ax, color="#2386C8", rot=35)
ax.set_title("Cropped faces per identity")
ax.set_xlabel(""); ax.set_ylabel("Images"); ax.set_ylim(0, crop_counts.max() * 1.15)
for i, count in enumerate(crop_counts):
    ax.text(i, count + 0.5, str(count), ha="center")
fig.tight_layout()
quality_show(fig, "class_counts.png")
display(cropped_quality[["width", "height", "brightness", "contrast", "sharpness"]].describe().round(2).T)

paired = clean_quality.merge(cropped_quality, left_on="path", right_on="source",
                            suffixes=("_full", "_crop"), validate="one_to_one")
comparison_metrics = pd.DataFrame({
    "retained_sources_mean": paired[["brightness_full", "contrast_full", "sharpness_full"]].mean().to_numpy(),
    "crops_mean": paired[["brightness_crop", "contrast_crop", "sharpness_crop"]].mean().to_numpy(),
}, index=["brightness", "contrast", "sharpness"])
display(comparison_metrics.round(2))
count_comparison.to_csv(CROP_REPORTS / "class_summary.csv")
comparison_metrics.to_csv(CROP_REPORTS / "before_after_quality.csv")

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, metric in zip(axes, ["brightness", "contrast", "sharpness"]):
    ax.hist(cropped_quality[metric], bins=25, edgecolor="white")
    ax.axvline(cropped_quality[metric].median(), color="darkorange", linestyle="--", label="Median")
    names = {"brightness": "Brightness (0-255)", "contrast": "Contrast (standard deviation)",
             "sharpness": "Sharpness (Laplacian variance)"}
    ax.set_title(names[metric]); ax.set_xlabel(names[metric]); ax.set_ylabel("Images"); ax.legend()
fig.tight_layout()
quality_show(fig, "quality_distributions.png")

display(Markdown(
    f"**Result:** {removed_now} image(s) removed this run; "
    f"{len(clean_quality)} images retained; **{len(cropped_quality)} crops** from "
    f"**{cropped_quality['class'].nunique()} classes**, all at 160 × 160. "
    f"Images per identity : {int(crop_counts.min())} to {int(crop_counts.max())}. "
    f"Median brightness : {cropped_quality.brightness.median():.1f}/255; "
    f"median contrast : {cropped_quality.contrast.median():.1f}; "
    f"median sharpness : {cropped_quality.sharpness.median():.1f}. "
    "Recognition performance must be evaluated separately."
))


display(Markdown("### Matched source and crop comparison"))
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, metric, title in zip(axes, ["brightness", "contrast", "sharpness"],
                           ["Mean brightness", "Mean contrast", "Mean sharpness"]):
    values = comparison_metrics.loc[metric].to_numpy()
    ax.bar(["Full images", "Crops 160 × 160"], values, color=["#8294a5", "#2386C8"])
    for i, value in enumerate(values):
        ax.text(i, value + max(values)*0.025, f"{value:.2f}", ha="center")
    ax.set_ylim(0, max(values)*1.2); ax.set_title(title)
fig.tight_layout()
quality_show(fig, "before_after_quality.png")

display(Markdown("### Crop quality by identity"))
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, metric, title in zip(axes, ["brightness", "contrast", "sharpness"],
                           ["Brightness", "Contrast", "Sharpness"]):
    groups = [cropped_quality.loc[cropped_quality['class'].eq(name), metric].to_numpy()
              for name in crop_counts.index]
    ax.boxplot(groups, tick_labels=crop_counts.index, patch_artist=True,
               boxprops={"facecolor": "#9fc4e8"})
    ax.set_title(title + " by identity"); ax.tick_params(axis="x", labelrotation=70, labelsize=8)
    ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
quality_show(fig, "quality_by_class.png")

display(Markdown(
    "**Interpretation:** the classes remain nearly balanced. "
    "Brightness increases after cropping, possibly because "
    "some dark background is removed. Contrast decreases in the face regions. "
    "Sharpness also changes with cropping and resizing; "
    "these scores are not directly comparable with the full-image threshold. "
    "These measurements do not demonstrate improved recognition."
))


In [ ]:
# Show ambiguous sources, then one example per identity.
ambiguous_sources = set(crop_status.loc[crop_status.valid_detections > 1, "path"])
gallery = pd.concat([
    crop_manifest.loc[crop_manifest.source.isin(ambiguous_sources)].head(8),
    crop_manifest.groupby("class", sort=True).head(1)
]).drop_duplicates("source").head(18)
fig, axes = plt.subplots((len(gallery)+5)//6, 6, figsize=(16, 3*((len(gallery)+5)//6)), squeeze=False)
for ax, (_, row) in zip(axes.flat, gallery.iterrows()):
    with Image.open(CROP_DIR / row.crop) as image:
        ax.imshow(image)
    ax.set_title(row["class"] + ("\n(multiple detections)" if row.source in ambiguous_sources else ""), fontsize=8)
    ax.axis("off")
for ax in list(axes.flat)[len(gallery):]:
    ax.axis("off")
fig.suptitle("Exported crops: one selected face per source")
fig.tight_layout()
quality_show(fig, "crop_gallery.png")

### Save and view the plots
Plots and tables appear inline. PNG copies are saved in `analyse/cropped_faces/reports/graphs`. To refresh only the crop analysis, run the two section 11.2 cells in order; they read existing crops without repeating filtering or detection.

## 11.3 Inspect images with multiple face detections
Review every retained source with multiple MTCNN detections. Secondary boxes may be real faces or false positives.

- Left: the source, selected box in green, and other boxes in orange, with detection IDs and confidence.
- Right: the saved crop, expected identity, and selected detection ID.

Check that the expected person was selected and the face is not cut off. Detection confidence does not verify identity. Pages contain five sources. Set `REVIEW_CLASS` to an identity to filter the gallery. This cell does not change crops or selections.

In [ ]:
from pathlib import Path
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image, ImageOps
from IPython.display import display, Markdown, Image as DisplayImage

REVIEW_CLASS = None  # Exemple : "Ariel_Sharon"
IMAGES_PER_PAGE = 5
assert IMAGES_PER_PAGE > 0
review_root = Path.cwd()
if not (review_root / "preprocessed_data").is_dir():
    review_root = review_root / "analyse"
review_data = review_root / "preprocessed_data"
review_crops = review_root / "cropped_faces"
review_reports = review_crops / "reports"
review_dir = review_reports / "multiface_review"
review_dir.mkdir(parents=True, exist_ok=True)
review_status = pd.read_csv(review_reports / "image_status.csv")
review_det = pd.read_csv(review_reports / "detections.csv")
review_manifest = pd.read_csv(review_reports / "crop_manifest.csv")
assert review_manifest.source.is_unique

review_index = review_status.loc[review_status.detections > 1].merge(
    review_manifest[["source", "crop", "detection_id", "selection_score", "confidence"]],
    left_on="path", right_on="source", how="left", validate="one_to_one"
).sort_values(["class", "path"]).reset_index(drop=True)
review_index.to_csv(review_dir / "review_index.csv", index=False)
review_visible = review_index if REVIEW_CLASS is None else review_index.loc[
    review_index["class"].eq(REVIEW_CLASS)].reset_index(drop=True)
display(Markdown(
    f"**{len(review_visible)} images to review** out of {len(review_index)} sources multidetections. "
    f"**{review_visible['class'].nunique()} classes** shown. Green = selected; orange = other detection."
))
display(review_visible.groupby("class").size().rename("images_multidetections").to_frame())

review_pages = []
for start in range(0, len(review_visible), IMAGES_PER_PAGE):
    page = review_visible.iloc[start:start + IMAGES_PER_PAGE]
    fig, axes = plt.subplots(len(page), 2, figsize=(12, 3.6 * len(page)), squeeze=False)
    for row_num, (_, item) in enumerate(page.iterrows()):
        original_ax, crop_ax = axes[row_num]
        src = review_data / item.path
        with Image.open(src) as image:
            original_ax.imshow(ImageOps.exif_transpose(image).convert("RGB"))
        faces = review_det.loc[review_det.path.eq(item.path)]
        chosen_id = None if pd.isna(item.detection_id) else int(item.detection_id)
        if chosen_id is not None:
            assert int((faces.detection_id == chosen_id).sum()) == 1
        for _, face in faces.iterrows():
            selected = chosen_id is not None and int(face.detection_id) == chosen_id
            color = "lime" if selected else "darkorange"
            x, y, w, h = [float(face[k]) for k in
                          ["clipped_x", "clipped_y", "clipped_width", "clipped_height"]]
            original_ax.add_patch(Rectangle((x, y), w, h, fill=False,
                edgecolor=color, linewidth=2.5 if selected else 1.6))
            original_ax.text(x, max(0, y-3),
                f"#{int(face.detection_id)} conf={face.confidence:.3f}" + (" CHOISI" if selected else ""),
                fontsize=7, color=color, va="bottom",
                bbox={"facecolor": "black", "alpha": .75, "pad": 1})
        original_ax.set_title(f"Source : {Path(item.path).name}\n{int(item.detections)} detections MTCNN", fontsize=9)
        if pd.notna(item.crop):
            with Image.open(review_crops / item.crop) as image:
                crop_ax.imshow(image.convert("RGB"))
            crop_ax.set_title(
                f"Expected identity : {item['class']}\nCrop #{chosen_id} — center and size score : {item.selection_score:.3f}",
                fontsize=9)
        else:
            crop_ax.text(.5, .5, "No crop exported", ha="center", va="center")
            crop_ax.set_title(f"Expected identity : {item['class']}", fontsize=9)
        original_ax.axis("off"); crop_ax.axis("off")
    page_no = start // IMAGES_PER_PAGE + 1
    fig.suptitle(f"Multiple-face review — page {page_no} — images {start+1} to {start+len(page)}", fontsize=13)
    fig.tight_layout(rect=(0, 0, 1, .98))
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", bbox_inches="tight", dpi=100)
    plt.close(fig)
    # Use distinct filenames for filtered galleries.
    prefix = "all" if REVIEW_CLASS is None else "class_" + str(REVIEW_CLASS).replace("/", "_").replace("\\", "_")
    output_path = review_dir / f"{prefix}_page_{page_no:02d}.png"
    output_path.write_bytes(buffer.getvalue())
    review_pages.append(output_path)
    display(Markdown(f"**Page {page_no} — images {start+1} to {start+len(page)}**"))
    display(DisplayImage(data=buffer.getvalue()))

if review_visible.empty:
    print("No multi-detection images for this filter.")
else:
    print(f"{len(review_visible)} images shown across {len(review_pages)} pages. Gallery saved in {review_dir.resolve()}")

# 12. Conclusions and next experiments
The cleaned dataset retains sharp images and backs up rejected files. Each exported source has one selected MTCNN crop. Inspect ambiguous selections and small faces before training.

The model-specific scripts align eyes from source landmarks. PCA uses normalized grayscale pixels; FaceNet uses RGB inputs and applies its expected normalization during embedding extraction. Evaluate macro F1 and per-identity scores on the shared split. Image quality statistics alone do not establish better recognition.

# 13. Split cropped_faces: 80% train / 20% test
Create a reproducible stratified split with `random_state=42`. Every identity appears in both subsets; rounding makes the proportions approximate.

Copy crops without modification into `cropped_faces_split/train/<class>/` and `cropped_faces_split/test/<class>/`. Keep the original crops. Reuse the same manifests for every model.

Check exact pixel duplicates and source dHash candidates at Hamming distance <= 5. A candidate crossing train and test blocks export for review. This heuristic cannot find every visually similar pair. Fit PCA and select hyperparameters inside training cross-validation. Reserve the test subset for final evaluation and keep augmentation in training.

In [ ]:
from pathlib import Path
import hashlib, io, json, shutil
from itertools import combinations
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from sklearn.model_selection import train_test_split
from IPython.display import display, Markdown, Image as DisplayImage

TEST_SIZE = 0.20
RANDOM_STATE = 42
NEAR_DUPLICATE_THRESHOLD = 5
split_base = Path.cwd()
if not (split_base / "cropped_faces").is_dir():
    split_base = split_base / "analyse"
crop_root = (split_base / "cropped_faces").resolve()
source_root = (split_base / "preprocessed_data").resolve()
SPLIT_DIR = (split_base / "cropped_faces_split").resolve()
assert SPLIT_DIR.parent == crop_root.parent and SPLIT_DIR != crop_root
split_images = pd.read_csv(crop_root / "reports" / "crop_manifest.csv")
assert split_images.source.is_unique and split_images.crop.is_unique
assert split_images.groupby("class").size().min() >= 2

train_rows, test_rows = train_test_split(
    split_images, test_size=TEST_SIZE, random_state=RANDOM_STATE,
    shuffle=True, stratify=split_images["class"])
split_manifest = split_images.copy()
split_manifest["split"] = "train"
split_manifest.loc[test_rows.index, "split"] = "test"
assert set(train_rows.source).isdisjoint(test_rows.source)
assert set(train_rows["class"]) == set(test_rows["class"]) == set(split_images["class"])

# Validate images before writing the split dataset.
def pixel_digest(image_path):
    with Image.open(image_path) as image:
        rgb = np.asarray(ImageOps.exif_transpose(image).convert("RGB"))
    return hashlib.sha256(str(rgb.shape).encode() + rgb.tobytes()).hexdigest()

source_hashes, crop_hashes, source_dhashes = [], [], []
for item in split_manifest.itertuples():
    crop_path = (crop_root / item.crop).resolve()
    source_path = (source_root / item.source).resolve()
    assert crop_path.is_relative_to(crop_root) and source_path.is_relative_to(source_root)
    crop_hashes.append(pixel_digest(crop_path))
    source_hashes.append(pixel_digest(source_path))
    with Image.open(source_path) as image:
        small = np.asarray(ImageOps.exif_transpose(image).convert("L").resize(
            (9, 8), Image.Resampling.LANCZOS))
    bits = small[:, 1:] > small[:, :-1]
    source_dhashes.append(int.from_bytes(np.packbits(bits.ravel()).tobytes(), "big"))
split_manifest["source_pixel_sha256"] = source_hashes
split_manifest["crop_pixel_sha256"] = crop_hashes
for column in ["source_pixel_sha256", "crop_pixel_sha256"]:
    assert split_manifest.groupby(column)["split"].nunique().max() == 1, (
        "Exact duplicate crosses train/test: create a grouped split.")

near_rows = []
for i, j in combinations(range(len(split_manifest)), 2):
    distance = (source_dhashes[i] ^ source_dhashes[j]).bit_count()
    if distance <= NEAR_DUPLICATE_THRESHOLD:
        a, b = split_manifest.iloc[i], split_manifest.iloc[j]
        near_rows.append({"source_a": a.source, "source_b": b.source,
                          "distance": distance, "split_a": a.split, "split_b": b.split,
                          "cross_split": a.split != b.split})
near_check = pd.DataFrame(near_rows, columns=[
    "source_a", "source_b", "distance", "split_a", "split_b", "cross_split"])
assert not near_check.cross_split.any(), (
    "Near-duplicate candidates cross train/test: review and create a grouped split.")

split_manifest["split_path"] = [
    (Path(item.split) / item.crop).as_posix() for item in split_manifest.itertuples()]
SPLIT_DIR.mkdir(parents=True, exist_ok=True)
old_manifest_path = SPLIT_DIR / "split_manifest.csv"
old_manifest = pd.read_csv(old_manifest_path) if old_manifest_path.exists() else pd.DataFrame()
for item in split_manifest.itertuples():
    src = (crop_root / item.crop).resolve()
    dst = (SPLIT_DIR / item.split_path).resolve()
    assert dst.is_relative_to(SPLIT_DIR / item.split)
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    assert src.read_bytes() == dst.read_bytes(), f"Copy differs : {dst}"

# On reruns, clean only previous copies recorded in the manifest.
if not old_manifest.empty:
    for rel in set(old_manifest.split_path) - set(split_manifest.split_path):
        target = (SPLIT_DIR / rel).resolve()
        assert target.is_relative_to(SPLIT_DIR) and Path(rel).parts[0] in {"train", "test"}
        if target.is_file():
            target.unlink()
split_manifest = split_manifest.sort_values(["split", "class", "source"]).reset_index(drop=True)
split_manifest.to_csv(old_manifest_path, index=False)
for split_name in ["train", "test"]:
    split_manifest.loc[split_manifest.split.eq(split_name)].to_csv(
        SPLIT_DIR / f"{split_name}.csv", index=False)
near_check.to_csv(SPLIT_DIR / "near_duplicate_check.csv", index=False)
split_counts = pd.crosstab(split_manifest["class"], split_manifest.split)[["train", "test"]]
split_counts["total"] = split_counts.train + split_counts.test
split_counts["test_percent"] = (100 * split_counts.test / split_counts.total).round(2)
split_counts.to_csv(SPLIT_DIR / "class_counts.csv")
settings = {"test_size_requested": TEST_SIZE, "random_state": RANDOM_STATE,
    "method": "stratified train_test_split with duplicate checks",
    "train_images": int((split_manifest.split == "train").sum()),
    "test_images": int((split_manifest.split == "test").sum()),
    "classes": int(split_manifest["class"].nunique()),
    "source_dataset": str(crop_root), "near_duplicate_threshold": NEAR_DUPLICATE_THRESHOLD,
    "near_duplicate_candidates": len(near_check)}
(SPLIT_DIR / "settings.json").write_text(json.dumps(settings, indent=2), encoding="utf-8")
display(Markdown(
    f"**Train : {settings['train_images']} images** — "
    f"**Test : {settings['test_images']} images** — **{settings['classes']} classes**. "
    f"Seed: {RANDOM_STATE}. No checked exact duplicate or dHash candidate crosses train and test."
))
display(split_counts)
if not near_check.empty:
    display(near_check)
fig, ax = plt.subplots(figsize=(12, 5))
split_counts[["train", "test"]].plot.bar(ax=ax, color=["#2386C8", "#F2A541"], rot=35)
ax.set_title("Stratified cropped_faces split: 80% train / 20% test")
ax.set_xlabel(""); ax.set_ylabel("Image count")
for bars in ax.containers:
    ax.bar_label(bars, padding=2, fontsize=8)
ax.set_ylim(0, split_counts.train.max() * 1.18)
fig.tight_layout()
buffer = io.BytesIO()
fig.savefig(buffer, format="png", dpi=120, bbox_inches="tight")
plt.close(fig)
(SPLIT_DIR / "class_distribution.png").write_bytes(buffer.getvalue())
display(DisplayImage(data=buffer.getvalue()))
print("Dataset split :", SPLIT_DIR)
print("Reuse train.csv and test.csv for every model.")